# 08 — Integration: Full End-to-End Research Query

This notebook brings together all CCA patterns:
- Hub-and-spoke coordinator
- Context isolation via explicit passing
- 4-5 scoped tools per agent
- Structured error handling
- Parallel + sequential task waves
- Deterministic conflict resolution

In [ ]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
sys.path.insert(0, str(Path('.').resolve()))

In [ ]:
from research_agents.models.research import SubTask
from research_agents.agent.coordinator import sort_tasks_into_waves, run_coordinator, build_research_report
from research_agents.agent.context_builder import build_subagent_context
from research_agents.agent.subagents import SUBAGENT_CONFIGS
from research_agents.data.scenarios import SCENARIOS
from research_agents.data.sources import SOURCE_RELIABILITY_RATINGS
from research_agents.models.research import SourceReliability
from tests.conftest import make_services

## Scenario: Remote Work Economic Impact

This scenario tests both conflict resolution AND error handling.

In [ ]:
scenario = SCENARIOS['economic_impact']
print(f'Query: {scenario.query}')
print(f'Expected agents: {scenario.expected_agents}')
print(f'Expected conflicts: {scenario.expected_conflicts}')
print(f'Expected gaps: {scenario.expected_gaps}')

## Step 1: Task Decomposition

In [ ]:
# Decompose into subtasks (normally the LLM does this)
tasks = [
    SubTask(task_id='web', agent_type='web_researcher',
        instruction='Search for studies on remote work and productivity',
        context='Focus on 2024 data. Look for both pro and con evidence.'),
    SubTask(task_id='data', agent_type='data_extractor',
        instruction='Query remote work statistics from the database',
        context='Use the remote_work_stats table'),
    SubTask(task_id='docs', agent_type='document_analyzer',
        instruction='Analyze the Stanford remote work study',
        context='Document ID: doc-remote-work-stanford'),
    SubTask(task_id='facts', agent_type='fact_checker',
        instruction='Verify productivity claims from web and document sources',
        context='Check claims about remote work productivity impact',
        depends_on=['web', 'docs']),
]

waves = sort_tasks_into_waves(tasks)
for i, wave in enumerate(waves):
    print(f'Wave {i}: {[t.task_id for t in wave]} ({"parallel" if len(wave) > 1 else "sequential"})' )

## Step 2: Context Isolation Check

Each subagent gets ONLY its explicit context.

In [ ]:
for task in tasks:
    ctx = build_subagent_context(task)
    print(f'{task.task_id} ({task.agent_type}):')
    print(f'  Context length: {len(ctx)} chars')
    print(f'  First 100 chars: {ctx[:100]}...')
    print()

## Step 3: Run Coordinator (Mock Mode)

Using mock client for reproducible results.

In [ ]:
from types import SimpleNamespace

call_count = 0
def mock_create(**kwargs):
    global call_count
    call_count += 1
    agent_responses = {
        1: 'Found 4 sources. McKinsey reports +13% productivity. Blog claims -20%.',
        2: 'Remote work stats: 9.4% fully remote, 18.2% hybrid in 2024.',
        3: 'Stanford study: hybrid workers 13% more productive, 24% higher satisfaction.',
        4: 'Verified: +13% claim supported by multiple sources. -20% claim unsupported.',
    }
    text = agent_responses.get(call_count, 'Analysis complete.')
    return SimpleNamespace(
        content=[SimpleNamespace(type='text', text=text)],
        stop_reason='end_turn',
        usage=SimpleNamespace(input_tokens=500, output_tokens=200),
    )

mock_client = SimpleNamespace(messages=SimpleNamespace(create=mock_create))
services = make_services()

results, waves = run_coordinator(mock_client, services, tasks)
for task_id, result in results.items():
    print(f'{task_id}: {result.content[:80]}...' if len(result.content) > 80 else f'{task_id}: {result.content}')

## Step 4: Conflict Resolution + Report

In [ ]:
# Build report with conflict resolution
conflicts = [{
    'claim': 'Remote workers are more productive',
    'sources_for': ['https://mckinsey.com/future-of-work', 'https://bls.gov/remote-work-stats'],
    'sources_against': ['https://workfromhome-blog.example.com/productivity'],
}]

reliability = {url: rel for url, rel in SOURCE_RELIABILITY_RATINGS.items()}

report = build_research_report(
    query=scenario.query,
    results=results,
    reliability_lookup=reliability,
    conflicts=conflicts,
    gaps=['https://timeout.example.com/remote-data (timeout)'],
)

print(f'Query: {report.query}')
print(f'Findings: {len(report.findings)}')
print(f'Conflicts resolved: {len(report.conflicts)}')
for c in report.conflicts:
    print(f'  - {c.claim}: {c.resolution} (confidence: {c.confidence:.2f})')
print(f'Gaps: {report.gaps}')
print(f'Confidence score: {report.confidence_score:.2f}')

## CCA Exam Tip

> The Multi-Agent Research System scenario draws from the three heaviest domains:
> - Agentic Architecture (27%)
> - Tool Design & MCP (18%)
> - Context Management & Reliability (15%)
>
> Together: **60% of the exam weight**. Master these patterns and you have a framework for every scenario.